# IRIS Quick Start

**Goal:** in <5 minutes, compile a small benchmark with all 3 variants
(QuComm baseline, IRIS-opt0 = UMS, IRIS-opt1 = UMS + EES) and compare
the **T_eff** and **L** metrics from the paper.

**Definitions** (IRIS paper, §5.5):
- `T_eff = #state_teleportations + 1.77 * #gate_teleportations`
- `L = total_execution_time` (seconds)

**Prereqs:** `conda activate iris-ae` (env from `setup.sh`). Run this notebook
from the artifact root or the `notebooks/` subdir.

> Cite: [Yun et al., *IRIS: A Compiler For Optimized Scheduling In Distributed Quantum Computers*, arXiv:2605.21795](https://arxiv.org/abs/2605.21795)


In [ ]:
import json, os, subprocess, sys, time, shutil
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'router' / '_iris.py').exists(), f'Run from artifact root or notebooks/, got {ROOT}'
print(f'artifact root: {ROOT}')

# Local IRIS-dataset checkout (benchmarks + mapping results).
# Recommended location: the repository root (./IRIS-dataset).
_ds = ROOT / 'IRIS-dataset'
if not _ds.exists():
    _ds = ROOT.parent / 'IRIS-dataset'
DATASET = Path(os.environ.get('IRIS_DATASET', _ds))

RE_CNOT_WEIGHT = 1.77
# results use the IRIS-dataset layout: <Mapping>/<Scheduling>/<bench>-<archdir>/
DS_MAPPER = {'ILP': 'MinCut', 'GCP-ILP': 'GCP-E', 'OEE-ILP': 'sOEE', 'WBCP': 'WBCP'}
DS_SCHED = {'QuComm': 'QuComm', 'IRIS-opt0': 'IRIS-noEES', 'IRIS-opt1': 'IRIS'}

def ensure_qasm(family, n):
    p = ROOT / 'bench' / family / f'{family}_n{n}.qasm'
    if not p.exists():
        raise FileNotFoundError(
            f'{p} missing — benchmarks are copied from the IRIS-dataset, not generated.\n'
            f'Run:  bash scripts/ensure_dataset.sh   (dataset expected at {DATASET})')
    return p

def seed_mapping_cache(bench, arch, results_dir):
    """Copy the mapping cache so the ILP mapper is skipped."""
    if not (DATASET / 'index.json').is_file():
        print(f'  (no IRIS-dataset at {DATASET}; the mapper will run from scratch)')
        return
    subprocess.run([sys.executable, str(ROOT / 'scripts' / 'seed_from_dataset.py'),
                    '--dataset', str(DATASET), '--results', str(results_dir),
                    '--benches', bench, '--archs', arch],
                   check=True, cwd=ROOT, stdout=subprocess.DEVNULL)

def run_one(variant_script, bench, arch, mapper, results_dir):
    env = {**os.environ, 'RESULTS_DIR': str(results_dir)}
    t0 = time.time()
    subprocess.run(['bash', str(ROOT / 'scripts' / f'run_{variant_script}.sh'),
                    bench, arch, mapper], check=True, cwd=ROOT, env=env,
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return time.time() - t0

def load_result(run_dir):
    res_paths = sorted(run_dir.glob('results*.json'))
    assert res_paths, f'no result JSON in {run_dir}'
    d = json.loads(res_paths[0].read_text())
    teff = d.get('num_state_teleportations', 0) + RE_CNOT_WEIGHT * d.get('num_gate_teleportations', 0)
    L = d.get('total_execution_time', 0.0)
    extra = run_dir / 'extra_opt.json'
    if extra.exists():
        ed = json.loads(extra.read_text())
        if ed.get('wall_time_ms_extra') is not None:
            L = ed['wall_time_ms_extra'] / 1000.0
    return {'T_eff': teff, 'L_s': L,
            'state_telep': d.get('num_state_teleportations', 0),
            'gate_telep': d.get('num_gate_teleportations', 0)}

ARCH_DIRS = {'F120': 'S40C5-2x2', 'F180': 'S42C5-2x3', 'F240': 'S40C5-3x3',
             'F500': 'S180C18-2x2', 'F800': 'S180C18-2x3', 'F1100': 'S180C18-3x3'}

def run_all_variants(bench, arch='F120', mapper='ILP', tag=''):
    rd = ROOT / 'results' / f'_quickstart{tag}'
    if rd.exists():
        shutil.rmtree(rd)
    seed_mapping_cache(bench, arch, rd)
    timings = {}
    for sh in ('qucomm', 'iris_opt0', 'iris_opt1'):
        dt = run_one(sh, bench, arch, mapper, rd)
        timings[sh] = dt
        print(f'  {sh:<14s} done in {dt:5.1f}s')
    archdir = ARCH_DIRS.get(arch, arch)
    rows = {}
    for v in ('QuComm', 'IRIS-opt0', 'IRIS-opt1'):
        rows[v] = load_result(rd / DS_MAPPER.get(mapper, mapper) / DS_SCHED[v] / f'{bench}-{archdir}')
    return rows, timings


## 1. First example: `bv_n120` (Bernstein–Vazirani, 120 qubits)

The paper's smallest F120 benchmark (~2–3 min for all three variants; the
mapping cache is imported from the IRIS-dataset, so the ILP mapper is
skipped). Expected: UMS cuts state
teleportations 73 → 60, and EES cuts latency a further ~1.5× vs QuComm.


In [ ]:
ensure_qasm('bv', 120)
print('\nRunning 3 variants on bv_n120 / F120 / ILP:')
rows_bv, _ = run_all_variants('bv_n120', tag='_bv')

print()
print(f"{'variant':<14s} {'T_eff':>8s}  {'L_s':>8s}  {'state_telep':>12s}  {'gate_telep':>11s}")
for v, r in rows_bv.items():
    print(f"{v:<14s} {r['T_eff']:8.1f}  {r['L_s']:8.4f}  {r['state_telep']:12d}  {r['gate_telep']:11d}")
qc = rows_bv['QuComm']
print(f'\nIRIS-opt1 latency speedup over QuComm: {qc["L_s"]/rows_bv["IRIS-opt1"]["L_s"]:.2f}x')


## 2. Denser example: `qaoa_3reg_n120` (3-regular QAOA, 120 qubits)

A denser interaction graph, so the UMS lookahead + foresight optimization has
more work to do (expect several minutes for the two IRIS variants). This is a
paper run: the same tuple appears in Table 5/6 (`MinCut` mapper, F120).


In [ ]:
ensure_qasm('qaoa_3reg', 120)
print('\nRunning 3 variants on qaoa_3reg_n120 / F120 / ILP:')
rows_qa, _ = run_all_variants('qaoa_3reg_n120', tag='_qa')

print()
print(f"{'variant':<14s} {'T_eff':>8s}  {'vs QuComm':>10s}  {'L_s':>8s}  {'vs QuComm':>10s}")
qc = rows_qa['QuComm']
for v, r in rows_qa.items():
    teff_ratio = r['T_eff'] / qc['T_eff'] if qc['T_eff'] else 1.0
    L_ratio = r['L_s'] / qc['L_s'] if qc['L_s'] else 1.0
    print(f"{v:<14s} {r['T_eff']:8.1f}  {teff_ratio:>9.1%}   {r['L_s']:8.4f}  {L_ratio:>9.1%}")

ums_red = (1 - rows_qa['IRIS-opt0']['T_eff'] / qc['T_eff']) * 100
ees_sp = qc['L_s'] / rows_qa['IRIS-opt1']['L_s']
print(f'\nUMS T_eff reduction:                 {ums_red:5.1f}%')
print(f'UMS+EES latency speedup vs QuComm:   {ees_sp:.2f}x')


## 3. Visualize the speedup (bar chart)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

variants = list(rows_qa.keys())
teff = [rows_qa[v]['T_eff'] for v in variants]
L    = [rows_qa[v]['L_s']   for v in variants]
colors = ['#888888', '#40B0C4', '#1C061A']

fig, (axT, axL) = plt.subplots(1, 2, figsize=(9, 3.2))
axT.bar(variants, teff, color=colors)
axT.set_title('T_eff  (lower is better)')
axT.set_ylabel('T_eff')
for i, v in enumerate(teff):
    axT.text(i, v, f'{v:.1f}', ha='center', va='bottom')

axL.bar(variants, L, color=colors)
axL.set_title('Execution latency L (s)  (lower is better)')
axL.set_ylabel('L (seconds)')
for i, v in enumerate(L):
    axL.text(i, v, f'{v:.4f}', ha='center', va='bottom')

fig.suptitle('qaoa_3reg_n120 on F120 (2x2 DQC, ILP mapper)')
fig.tight_layout()
plt.show()


## 4. Next steps

- **Get the dataset**: download `IRIS-dataset.tar` (~1.4 GB) from
  [zenodo.org/records/22152933](https://zenodo.org/records/22152933) and `tar -xf` it
- **Import everything once**: `python scripts/seed_from_dataset.py --dataset /path/to/IRIS-dataset`
  (copies all benchmarks + all mapping caches into `results/_full/`)
- **Full reproduce**: `RESULTS_DIR=$PWD/results/_full bash scripts/reproduce_all.sh` (hours — F120 + F180 + F240 sweep)
- **Per-paper outputs**: `RESULTS_BASE=$PWD/results/_full bash data_generator/run_all.sh` after the sweep completes
- **Tests**: `python tests/test_ees_postcondition.py` and `python tests/verify_extra_opt.py --root results/_full`

See [`README.md`](../README.md) for the full reproduction guide.
